In [23]:
import polars as pl

# Just creating a fake DataFrame to experiment with
training_data = pl.DataFrame({'age': [22, 32, 23, 38, 32, 55, 40], 'salary': [120000, 100000, 80000, 45000, 90000, 75000, 55000]})
print('Training')
display(training_data)
print('\nTest')
test_data = pl.DataFrame({'age': [33, 42, 25], 'salary': [115000, 143000, 53000]})
test_data

Training


age,salary
i64,i64
22,120000
32,100000
23,80000
38,45000
32,90000
55,75000
40,55000



Test


age,salary
i64,i64
33,115000
42,143000
25,53000


## Using Scikit-Learn

In [24]:
# If we need to do some kind of distance-based algorithm with these columns, it would
# help to transform them to the same scale first.
#
# Standard scaling will make the values center around 0.0 and show the variance by
# standard deviation
#
# SciKit-Learn has a built-in way to do this for us:
from sklearn.preprocessing import StandardScaler

# Create an instance of the transform object we're going to use
scaler = StandardScaler()

scaled_train = scaler.fit_transform(training_data)
scaled_test = scaler.transform(test_data)
print("Training")
display(pl.DataFrame(scaled_train, schema=training_data.columns))
print("\nTest")
display(pl.DataFrame(scaled_test, schema=test_data.columns))

Training


age,salary
f64,f64
-1.203561,1.64934
-0.246183,0.809676
-1.107823,-0.029988
0.328244,-1.4994
-0.246183,0.389844
1.955787,-0.239904
0.51972,-1.079568



Test


age,salary
f64,f64
-0.150445,1.439424
0.711195,2.614954
-0.916348,-1.163535


## Native Polars

In [25]:
# A Z-score Normalization scaler is given by this formula:
#
#    (value - mean)
# -------------------
# (standard deviation)
#
# We can use polars native expressions to create columns with their normalized
# counterpart

columns_to_scale = ['age', 'salary']
train_means = {}
train_stds = {}
for column in columns_to_scale:
  train_means[column] = training_data.select(pl.col(column)).to_series().mean()
  train_stds[column] = training_data.select(pl.col(column)).to_series().std(ddof=0)

training_scaled = training_data
testing_scaled = test_data

for column in columns_to_scale:
  training_scaled = training_scaled.with_columns(((pl.col(column) - train_means[column]) / train_stds[column]).name.suffix('_scaled'))
  testing_scaled = testing_scaled.with_columns(((pl.col(column) - train_means[column]) / train_stds[column]).name.suffix('_scaled'))

print("Training")
display(training_scaled)
print("\nTest")
display(testing_scaled)

Training


age,salary,age_scaled,salary_scaled
i64,i64,f64,f64
22,120000,-1.203561,1.64934
32,100000,-0.246183,0.809676
23,80000,-1.107823,-0.029988
38,45000,0.328244,-1.4994
32,90000,-0.246183,0.389844
55,75000,1.955787,-0.239904
40,55000,0.51972,-1.079568



Test


age,salary,age_scaled,salary_scaled
i64,i64,f64,f64
33,115000,-0.150445,1.439424
42,143000,0.711195,2.614954
25,53000,-0.916348,-1.163535
